In [1]:
!nvidia-smi
!nvcc -V

Mon Jun  9 08:27:26 2025       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 550.54.15              Driver Version: 550.54.15      CUDA Version: 12.4     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
%cd /content
#サブモジュールを含めてリポジトリをクローンする
!git clone https://github.com/graphdeco-inria/gaussian-splatting --recursive

/content
Cloning into 'gaussian-splatting'...
remote: Enumerating objects: 859, done.
remote: Total 859 (delta 0), reused 0 (delta 0), pack-reused 859 (from 1)
Receiving objects: 100% (859/859), 78.64 MiB | 62.32 MiB/s, done.
Resolving deltas: 100% (492/492), done.
Submodule 'SIBR_viewers' (https://gitlab.inria.fr/sibr/sibr_core.git) registered for path 'SIBR_viewers'
Submodule 'submodules/diff-gaussian-rasterization' (https://github.com/graphdeco-inria/diff-gaussian-rasterization.git) registered for path 'submodules/diff-gaussian-rasterization'
Submodule 'submodules/fused-ssim' (https://github.com/rahul-goel/fused-ssim.git) registered for path 'submodules/fused-ssim'
Submodule 'submodules/simple-knn' (https://gitlab.inria.fr/bkerbl/simple-knn.git) registered for path 'submodules/simple-knn'
Cloning into '/content/gaussian-splatting/SIBR_viewers'...
remote: Enumerating objects: 3293, done.        
remote: Counting objects: 100% (322/322), done.        
remote: Compressing objects: 100%

In [4]:
# CUDA 11.8 install
!wget https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
!sudo sh cuda_11.8.0_520.61.05_linux.run --silent --toolkit

# 環境変数の設定
import os
os.environ["PATH"] = "/usr/local/cuda-11.8/bin:" + os.environ["PATH"]
os.environ["LD_LIBRARY_PATH"] = "/usr/local/cuda-11.8/lib64:" + os.environ.get("LD_LIBRARY_PATH", "")

!nvcc -V

--2025-06-09 08:28:13--  https://developer.download.nvidia.com/compute/cuda/11.8.0/local_installers/cuda_11.8.0_520.61.05_linux.run
Resolving developer.download.nvidia.com (developer.download.nvidia.com)... 23.59.88.207, 23.59.88.195
Connecting to developer.download.nvidia.com (developer.download.nvidia.com)|23.59.88.207|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 4336730777 (4.0G) [application/octet-stream]
Saving to: ‘cuda_11.8.0_520.61.05_linux.run’

cuda_11.8.0_520.61. 100%[===================>]   4.04G   295MB/s    in 13s     

2025-06-09 08:28:26 (320 MB/s) - ‘cuda_11.8.0_520.61.05_linux.run’ saved [4336730777/4336730777]

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2022 NVIDIA Corporation
Built on Wed_Sep_21_10:33:58_PDT_2022
Cuda compilation tools, release 11.8, V11.8.89
Build cuda_11.8.r11.8/compiler.31833905_0


In [5]:
# いったん古いPyTorchを削除
!pip uninstall -y torch torchvision torchaudio

# PyTorch 2.0.1 + CUDA 11.8 をインストール（安定版）
!pip install torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118

Found existing installation: torch 2.6.0+cu124
Uninstalling torch-2.6.0+cu124:
  Successfully uninstalled torch-2.6.0+cu124
Found existing installation: torchvision 0.21.0+cu124
Uninstalling torchvision-0.21.0+cu124:
  Successfully uninstalled torchvision-0.21.0+cu124
Found existing installation: torchaudio 2.6.0+cu124
Uninstalling torchaudio-2.6.0+cu124:
  Successfully uninstalled torchaudio-2.6.0+cu124
Looking in indexes: https://download.pytorch.org/whl/cu118
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.2/23.2 MB 105.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 875.6/875.6 kB 58.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.1/13.1 MB 127.2 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 663.9/663.9 MB 2.1 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 417.9/417.9 MB 2.9 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 168.4/168.4 MB 15.4 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5

In [6]:
!pip uninstall fastai -y
!pip install fastai==2.7.12
!pip show torch fastai

Found existing installation: fastai 2.7.19
Uninstalling fastai-2.7.19:
  Successfully uninstalled fastai-2.7.19
INFO: pip is looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of torchvision to determine which version is compatible with other requirements. This could take a while.
INFO: This is taking longer than usual. You might need to provide the dependency resolver with stricter constraints to reduce runtime. See https://pip.pypa.io/warnings/backtracking for guidance. If you want to abort this run, press Ctrl + C.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 233.1/233.1 kB 10.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 77.2/77.2 kB 7.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 619.9/619.9 MB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 317.1/317.1 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━

In [7]:
# 保存したいパスと内容を指定
file_path = "/content/gaussian-splatting/requirements.txt"
content = "plyfile=0.8.1"

# ファイルの作成と書き込み
with open(file_path, "w") as f:
    f.write(content)

print(f"{file_path} にファイルを保存しました。")


/content/gaussian-splatting/requirements.txt にファイルを保存しました。


plyfile==0.8.1
tqdm
submodules/diff-gaussian-rasterization
submodules/simple-knn

In [8]:

%cd /content/gaussian-splatting
!pip install -r requirements.txt

/content/gaussian-splatting
Processing ./submodules/diff-gaussian-rasterization
  Preparing metadata (setup.py) ... done
Processing ./submodules/simple-knn
  Preparing metadata (setup.py) ... done
  Created wheel for diff_gaussian_rasterization: filename=diff_gaussian_rasterization-0.0.0-cp311-cp311-linux_x86_64.whl size=2974245 sha256=52f18a8f5723c20edab8f23f05d21f43b00be08243703ffe37f11cd1ec80deb4
  Stored in directory: /root/.cache/pip/wheels/d3/71/6a/2819f27db685a5ef06993166d0a6a1efb020245603f2f33da7
  Created wheel for simple_knn: filename=simple_knn-0.0.0-cp311-cp311-linux_x86_64.whl size=2904820 sha256=f5a140e10316f02715840ba3ed05e3051d28dd81e6ae833def88f6bb4e1a7897
  Stored in directory: /root/.cache/pip/wheels/db/0b/36/0f52647369b0e045f9a502b93e62051a2085d404ce211bead6
Successfully built diff_gaussian_rasterization simple_knn


In [ ]:
!rm -rf /content/3dgs

In [ ]:
!rm -rf /content/3dgs1

In [9]:
# Drive からフォルダごとコピー（-r は再帰的）
!cp -r /content/drive/MyDrive/court/input /content/3dgs/
!cp -r /content/drive/MyDrive/court/sparse /content/3dgs/

In [ ]:
# Drive からフォルダごとコピー（-r は再帰的）
!cp -r /content/drive/MyDrive/colab/gaussian_splatting_project/3dgs_split1 /content/3dgs1/
!cp -r /content/drive/MyDrive/colab/gaussian_splatting_project/3dgs_split2 /content/3dgs2/

In [ ]:
!cp -r /content/drive/MyDrive/colab/gaussian_splatting_project/3dgs_split1 /content/3dgs1/

In [ ]:
!cp -r /content/drive/MyDrive/colmap_splits/3dgs_split1 /content/3dgs1/
!cp -r /content/drive/MyDrive/colmap_splits/3dgs_split2 /content/3dgs2/

In [ ]:
# Drive からフォルダごとコピー（-r は再帰的）
!cp -r /content/drive/MyDrive/colab/gaussian_splatting_project/input /content/3dgs/
!cp -r /content/drive/MyDrive/colab/gaussian_splatting_project/sparse /content/3dgs/

In [ ]:
!cp -r /content/drive/MyDrive/backup_3dgs /content/3dgs/

In [10]:
!apt-get update
!apt-get install -y \
    git cmake ninja-build build-essential \
    libboost-program-options-dev libboost-filesystem-dev \
    libboost-graph-dev libboost-system-dev libboost-test-dev \
    libeigen3-dev libflann-dev libfreeimage-dev libmetis-dev \
    libgoogle-glog-dev libgflags-dev libsqlite3-dev libglew-dev \
    qtbase5-dev libqt5opengl5-dev libcgal-dev libceres-dev


Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Hit:3 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:4 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:5 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [1,765 kB]
Get:6 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:7 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Get:8 https://r2u.stat.illinois.edu/ubuntu jammy/main amd64 Packages [2,740 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:10 http://archive.ubuntu.com/ubuntu jammy-updates/universe amd64 Packages [1,553 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Get:13 http://archive.ubuntu.com/ubu

In [11]:
!git clone https://github.com/colmap/colmap.git
%cd colmap
!git checkout dev
!mkdir build
%cd build
!cmake .. -DCUDA_ENABLED=ON -DCMAKE_CUDA_ARCHITECTURES="75" -GNinja
!ninja
!sudo ninja install

Cloning into 'colmap'...
remote: Enumerating objects: 27950, done.
remote: Counting objects: 100% (512/512), done.
remote: Compressing objects: 100% (293/293), done.
remote: Total 27950 (delta 393), reused 225 (delta 219), pack-reused 27438 (from 2)
Receiving objects: 100% (27950/27950), 71.06 MiB | 45.88 MiB/s, done.
Resolving deltas: 100% (21556/21556), done.
/content/gaussian-splatting/colmap
error: pathspec 'dev' did not match any file(s) known to git
/content/gaussian-splatting/colmap/build
-- Enabling LSD support
-- The C compiler identification is GNU 11.4.0
-- The CXX compiler identification is GNU 11.4.0
-- Detecting C compiler ABI info
-- Detecting C compiler ABI info - done
-- Check for working C compiler: /usr/bin/cc - skipped
-- Detecting C compile features
-- Detecting C compile features - done
-- Detecting CXX compiler ABI info
-- Detecting CXX compiler ABI info - done
-- Check for working CXX compiler: /usr/bin/c++ - skipped
-- Detecting CXX compile features
-- Detectin

In [12]:
!colmap -h

COLMAP 3.12.0.dev0 -- Structure-from-Motion and Multi-View Stereo
(Commit 3e3ecb16 on 2025-06-06 with CUDA)

Usage:
  colmap [command] [options]
Documentation:
  https://colmap.github.io/
Example usage:
  colmap help [ -h, --help ]
  colmap gui
  colmap gui -h [ --help ]
  colmap automatic_reconstructor -h [ --help ]
  colmap automatic_reconstructor --image_path IMAGES --workspace_path WORKSPACE
  colmap feature_extractor --image_path IMAGES --database_path DATABASE
  colmap exhaustive_matcher --database_path DATABASE
  colmap mapper --image_path IMAGES --database_path DATABASE --output_path MODEL
  ...
Available commands:
  help
  gui
  automatic_reconstructor
  bundle_adjuster
  color_extractor
  database_cleaner
  database_creator
  database_merger
  delaunay_mesher
  exhaustive_matcher
  feature_extractor
  feature_importer
  hierarchical_mapper
  image_deleter
  image_filterer
  image_rectifier
  image_registrator
  image_undistorter
  image_undistorter_standalone
  mapper
  match

In [13]:
!pip install numpy==1.24.4 --force-reinstall
!pip install jax==0.4.20 jaxlib==0.4.20 --force-reinstall
!pip install tensorflow==2.13.0 --force-reinstall

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 17.3/17.3 MB 124.1 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
jax 0.5.2 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
treescope 0.1.9 requires numpy>=1.25.2, but you have numpy 1.24.4 which is incompatible.
jaxlib 0.5.1 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
pymc 5.23.0 requires numpy>=1.25.0, but you have numpy 1.24.4 which is incompatible.
blosc2 3.3.4 requires numpy>=1.26, but you have numpy 1.24.4 which is incompatible.
xarray-einstats 0.9.0 requires numpy>=1.25, but you have numpy 1.24.4 which is incompatible.
thinc 8.3.6 requires numpy<3.0.0,>=2.0.0, but you have numpy 1.24.4 which is inco

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.1/62.1 kB 4.5 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 62.0/62.0 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╸ 85.8/85.8 MB 221.3 MB/s eta 0:00:01^C
^C


In [4]:
import numpy as np
print(np.__version__)

1.24.4


memo: 600장의 input -> convert통과 후 train에서 시스템RAM이 용량 초과

In [2]:
%cd /content/gaussian-splatting/
!python convert.py -s /content/3dgs

/content/gaussian-splatting
I0609 08:50:45.036237 15261 misc.cc:44] 
Feature extraction
I0609 08:50:45.048410 15270 sift.cc:721] Creating SIFT GPU feature extractor
I0609 08:50:45.386327 15271 feature_extraction.cc:259] Processed file [1/301]
I0609 08:50:45.386371 15271 feature_extraction.cc:262]   Name:            00001.jpg
I0609 08:50:45.386376 15271 feature_extraction.cc:271]   Dimensions:      1920 x 1080
I0609 08:50:45.386381 15271 feature_extraction.cc:274]   Camera:          #1 - OPENCV
I0609 08:50:45.386386 15271 feature_extraction.cc:277]   Focal Length:    1152.00px (Prior)
I0609 08:50:45.386396 15271 feature_extraction.cc:281]   Features:        8899
I0609 08:50:45.453701 15271 feature_extraction.cc:259] Processed file [2/301]
I0609 08:50:45.453733 15271 feature_extraction.cc:262]   Name:            00002.jpg
I0609 08:50:45.453737 15271 feature_extraction.cc:271]   Dimensions:      1920 x 1080
I0609 08:50:45.453739 15271 feature_extraction.cc:274]   Camera:          #1 - OPE

In [6]:
%cd /content/gaussian-splatting/
!python train.py \
  --source_path /content/3dgs \
  --model_path /content/3dgs/test \
  --iterations 500 \
  --save_iterations 100 \
  --checkpoint_iterations 100 500 \
  --data_device cuda \
  --percent_dense 0.001 \
  --white_background

/content/gaussian-splatting
Optimizing /content/3dgs/test
Output folder: /content/3dgs/test [09/06 09:27:00]
Tensorboard disabled: logging is off [09/06 09:27:00]
Reading camera 301/301 [09/06 09:27:02]
Converting point3d.bin to .ply, will happen only the first time you open the scene. [09/06 09:27:02]
Loading Training Cameras [09/06 09:27:03]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [09/06 09:27:03]
Loading Test Cameras [09/06 09:27:27]
Number of points at initialisation :  166079 [09/06 09:27:27]
Training progress:  20% 100/500 [00:23<01:30,  4.41it/s, Loss=0.2481639, Depth Loss=0.0000000]
[ITER 100] Saving Gaussians [09/06 09:27:51]

[ITER 100] Saving Checkpoint [09/06 09:27:52]
Training progress: 100% 500/500 [01:44<00:00,  4.79it/s, Loss=0.1784772, Depth Loss=0.0000000]

[ITER 500] Saving Gaussians [09/06 09:29:12]

[ITER 500] Saving Checkpoint [09/06 09:29:13]



In [7]:
%cd /content/gaussian-splatting/
!python train.py \
  --source_path /content/3dgs \
  --model_path /content/3dgs/output \
  --iterations 50000 \
  --save_iterations 10000 \
  --data_device cuda \
  --percent_dense 0.001 \
  --white_background

/content/gaussian-splatting
Optimizing /content/3dgs/output
Output folder: /content/3dgs/output [09/06 09:29:39]
Tensorboard disabled: logging is off [09/06 09:29:39]
Reading camera 301/301 [09/06 09:29:41]
Loading Training Cameras [09/06 09:29:41]
[ INFO ] Encountered quite large input images (>1.6K pixels width), rescaling to 1.6K.
 If this is not desired, please explicitly specify '--resolution/-r' as 1 [09/06 09:29:41]
Loading Test Cameras [09/06 09:30:05]
Number of points at initialisation :  166079 [09/06 09:30:05]
Training progress:  14% 7000/50000 [23:53<2:50:38,  4.20it/s, Loss=0.0984851, Depth Loss=0.0000000]
[ITER 7000] Evaluating train: L1 0.04639021009206772 PSNR 23.320557785034183 [09/06 09:53:58]
Training progress:  20% 10000/50000 [36:21<2:52:31,  3.86it/s, Loss=0.0753982, Depth Loss=0.0000000]
[ITER 10000] Saving Gaussians [09/06 10:06:26]
Training progress:  60% 30000/50000 [2:02:46<1:19:39,  4.18it/s, Loss=0.0471287, Depth Loss=0.0000000]
[ITER 30000] Evaluating trai

In [ ]:
!python train.py \
  --source_path /content/3dgs \
  --model_path /content/3dgs/output \
  --iterations 50000 \
  --data_device cuda \
  --percent_dense 0.001 \
  --white_background \
  --start_checkpoint /content/3dgs/output/chkpnt14000.pth

2025-04-23 01:58:53.785677: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745373533.805610   63957 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745373533.811708   63957 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-23 01:58:53.832553: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/3dgs/output
Output folder: /content/3dgs/output [23/04 01:58:57]
Reading camera 596/596 [23/04 01:58:59]


-> 사진을 298,298으로 나눠서 convert, train

In [ ]:
import os
import shutil

# 元画像ディレクトリ
original_dir = "/content/3dgs/input"

# 分割後の出力ディレクトリ
split_dirs = ["/content/3dgs_split1/input", "/content/3dgs_split2/input"]

# 画像ファイル一覧を取得（ソートして時系列順と仮定）
image_files = sorted([f for f in os.listdir(original_dir) if f.lower().endswith(('.jpg', '.png', '.jpeg'))])

# 総枚数と分割数
total = len(image_files)
split_point = total // 2  # 2分割

# 分割実行
os.makedirs(split_dirs[0], exist_ok=True)
os.makedirs(split_dirs[1], exist_ok=True)

for i, f in enumerate(image_files):
    src = os.path.join(original_dir, f)
    if i < split_point:
        dst = os.path.join(split_dirs[0], f)
    else:
        dst = os.path.join(split_dirs[1], f)
    shutil.copy2(src, dst)

split_counts = [len(os.listdir(split_dirs[0])), len(os.listdir(split_dirs[1]))]
split_dirs, split_counts


(['/content/3dgs_split1/input', '/content/3dgs_split2/input'], [298, 298])

In [ ]:
%cd /content/gaussian-splatting/
!python convert.py -s /content/3dgs1

/content/gaussian-splatting
I0419 21:18:56.063261 27524 misc.cc:44] 
Feature extraction
I0419 21:18:56.063995 27527 sift.cc:721] Creating SIFT GPU feature extractor
I0419 21:18:56.393256 27528 feature_extraction.cc:258] Processed file [1/298]
I0419 21:18:56.393301 27528 feature_extraction.cc:261]   Name:            output_0001.png
I0419 21:18:56.393308 27528 feature_extraction.cc:270]   Dimensions:      1280 x 720
I0419 21:18:56.393316 27528 feature_extraction.cc:273]   Camera:          #1 - OPENCV
I0419 21:18:56.393322 27528 feature_extraction.cc:276]   Focal Length:    1536.00px
I0419 21:18:56.393383 27528 feature_extraction.cc:280]   Features:        5985
I0419 21:18:56.441296 27528 feature_extraction.cc:258] Processed file [2/298]
I0419 21:18:56.441334 27528 feature_extraction.cc:261]   Name:            output_0002.png
I0419 21:18:56.441342 27528 feature_extraction.cc:270]   Dimensions:      1280 x 720
I0419 21:18:56.441349 27528 feature_extraction.cc:273]   Camera:          #1 - O

In [ ]:
!python convert.py -s /content/3dgs2

I0419 22:21:28.851497 44268 misc.cc:44] 
Feature extraction
I0419 22:21:28.852125 44271 sift.cc:721] Creating SIFT GPU feature extractor
I0419 22:21:29.145408 44272 feature_extraction.cc:258] Processed file [1/298]
I0419 22:21:29.145439 44272 feature_extraction.cc:261]   Name:            output_0299.png
I0419 22:21:29.145445 44272 feature_extraction.cc:270]   Dimensions:      1280 x 720
I0419 22:21:29.145485 44272 feature_extraction.cc:273]   Camera:          #1 - OPENCV
I0419 22:21:29.145499 44272 feature_extraction.cc:276]   Focal Length:    1536.00px
I0419 22:21:29.145525 44272 feature_extraction.cc:280]   Features:        8368
I0419 22:21:29.191620 44272 feature_extraction.cc:258] Processed file [2/298]
I0419 22:21:29.191661 44272 feature_extraction.cc:261]   Name:            output_0300.png
I0419 22:21:29.191671 44272 feature_extraction.cc:270]   Dimensions:      1280 x 720
I0419 22:21:29.191679 44272 feature_extraction.cc:273]   Camera:          #1 - OPENCV
I0419 22:21:29.191686 

In [ ]:
!rm -rf /content/3dgs1/test/

In [ ]:
%cd /content/gaussian-splatting/
!python train.py \
  --source_path /content/3dgs1 \
  --model_path /content/3dgs1/test \
  --iterations 500 \
  --save_iterations 100 \
  --checkpoint_iterations 100 500 \
  --data_device cuda \
  --percent_dense 0.001 \
  --white_background

/content/gaussian-splatting
2025-04-20 05:26:33.491530: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745126793.512006   23186 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745126793.518123   23186 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-20 05:26:33.538916: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/3dgs1/test
Output folder: /content/3dgs1/test [20/04 05:26:37]
Reading camera

In [ ]:
%cd /content/gaussian-splatting/
!python train.py \
  --source_path /content/3dgs1 \
  --model_path /content/3dgs1/output \
  --iterations 50000 \
  --save_iterations 100 \
  --data_device cuda \
  --percent_dense 0.001 \
  --white_background

/content/gaussian-splatting
2025-04-20 05:39:57.740627: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745127597.761341   26666 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745127597.767941   26666 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-20 05:39:57.788677: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/3dgs1/output
Output folder: /content/3dgs1/output [20/04 05:40:01]
Reading ca

In [ ]:
%cd /content/gaussian-splatting/
!python train.py \
  --source_path /content/3dgs2 \
  --model_path /content/3dgs2/output \
  --iterations 50000 \
  --save_iterations 100 \
  --data_device cuda \
  --percent_dense 0.001 \
  --white_background

/content/gaussian-splatting
2025-04-20 07:11:08.507802: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745133068.528840   50093 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745133068.535432   50093 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-20 07:11:08.569199: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/3dgs2/output
Output folder: /content/3dgs2/output [20/04 07:11:14]
Reading ca

In [ ]:
!ls /content/3dgs1/test

cameras.json  events.out.tfevents.1745126191.a1a1837c6555.20559.0  input.ply
cfg_args      exposure.json					   point_cloud


marge해서 얻은 ply파일 잘 안됨 -> 450장으로 한번에 실행

In [ ]:
%cd /content/gaussian-splatting/
!python convert.py -s /content/3dgs

/content/gaussian-splatting
I0420 23:36:01.062467 22771 misc.cc:44] 
Feature extraction
I0420 23:36:01.063095 22774 sift.cc:721] Creating SIFT GPU feature extractor
I0420 23:36:01.325819 22775 feature_extraction.cc:258] Processed file [1/447]
I0420 23:36:01.325939 22775 feature_extraction.cc:261]   Name:            frame_001.png
I0420 23:36:01.325953 22775 feature_extraction.cc:270]   Dimensions:      1280 x 720
I0420 23:36:01.325963 22775 feature_extraction.cc:273]   Camera:          #1 - OPENCV
I0420 23:36:01.325970 22775 feature_extraction.cc:276]   Focal Length:    1536.00px
I0420 23:36:01.325990 22775 feature_extraction.cc:280]   Features:        5985
I0420 23:36:01.393847 22775 feature_extraction.cc:258] Processed file [2/447]
I0420 23:36:01.393889 22775 feature_extraction.cc:261]   Name:            frame_002.png
I0420 23:36:01.393898 22775 feature_extraction.cc:270]   Dimensions:      1280 x 720
I0420 23:36:01.393901 22775 feature_extraction.cc:273]   Camera:          #1 - OPENC

In [ ]:
%cd /content/gaussian-splatting/
!python train.py \
  --source_path /content/3dgs \
  --model_path /content/3dgs/test \
  --iterations 500 \
  --save_iterations 100 \
  --checkpoint_iterations 100 500 \
  --data_device cuda \
  --percent_dense 0.001 \
  --white_background

/content/gaussian-splatting
2025-04-21 13:35:20.835742: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745242521.078299   25235 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745242521.148933   25235 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-21 13:35:21.651436: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/3dgs/test
Output folder: /content/3dgs/test [21/04 13:35:29]
Reading camera 4

In [ ]:
%cd /content/gaussian-splatting/
!python train.py \
  --source_path /content/3dgs \
  --model_path /content/3dgs/output \
  --iterations 50000 \
  --save_iterations 10000 \
  --data_device cuda \
  --percent_dense 0.001 \
  --white_background

/content/gaussian-splatting
2025-04-21 13:29:53.386202: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1745242193.650967   23840 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1745242193.723978   23840 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-04-21 13:29:54.280158: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
Optimizing /content/3dgs/output
Output folder: /content/3dgs/output [21/04 13:30:02]
Reading came

In [8]:
from google.colab import drive
import shutil

# 保存先（任意で名前変えてOK）
backup_path = "/content/drive/MyDrive/court/3dgsOutput"

# コピー実行
shutil.copytree("/content/3dgs/output", backup_path)
print("✅ /content/3dgs が Drive に保存されました！")

✅ /content/3dgs が Drive に保存されました！


In [ ]:
from google.colab import drive
import shutil

# 保存先のベースディレクトリ
drive_base = "/content/drive/MyDrive/colmap_splits"

# コピー元・コピー先
shutil.copytree("/content/3dgs1", os.path.join(drive_base, "3dgs_split1"))
shutil.copytree("/content/3dgs2", os.path.join(drive_base, "3dgs_split2"))

print("✅ Google Drive に保存完了！")

FileExistsError: [Errno 17] File exists: '/content/drive/MyDrive/colmap_splits/3dgs_split1'

In [ ]:
from google.colab import drive
import shutil

# 保存先のベースディレクトリ
drive_base = "/content/drive/MyDrive/colmap_splits"

# コピー元・コピー先
shutil.copytree("/content/3dgs1", os.path.join(drive_base, "3dgs_split1"))
shutil.copytree("/content/3dgs2", os.path.join(drive_base, "3dgs_split2"))

print("✅ Google Drive に保存完了！")

✅ Google Drive に保存完了！


In [ ]:
!pip install open3d -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 447.7/447.7 MB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.0/8.0 MB 45.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.7/101.7 kB 9.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 139.8/139.8 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 228.0/228.0 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 33.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 30.1 MB/s eta 0:00:00


In [ ]:
# セッションがリセットされたので、再度必要なモジュールをインポートしてマージ処理を実行します
import open3d as o3d

# 再定義：マージ対象と出力パス
ply_path_1 = "/content/3dgs1/output/point_cloud/iteration_50000/point_cloud.ply"
ply_path_2 = "/content/3dgs2/output/point_cloud/iteration_50000/point_cloud.ply"
output_merged_path = "/content/merged_point_cloud.ply"

# 点群読み込み
pcd1 = o3d.io.read_point_cloud(ply_path_1)
pcd2 = o3d.io.read_point_cloud(ply_path_2)

# 結合
pcd_combined = pcd1 + pcd2

# 保存
o3d.io.write_point_cloud(output_merged_path, pcd_combined)

output_merged_path


'/content/merged_point_cloud.ply'